# 1. Importación de librerias

In [2]:
from mapie.metrics import regression_coverage_score, regression_coverage_score_v2, regression_mean_width_score
from mapie.regression import MapieQuantileRegressor

from utils.transformations import ExtendedTransformation, SimpleTransformation
from utils.filters import SimpleFilter
import pandas as pd
import numpy as np

# 2. Preprocesamiento de los datos

In [3]:
df_train = pd.read_csv("data/preprocessed/train_data.csv")
X_train, y_train = df_train.drop(columns=['Price']), df_train[['Price']]
preprocessor = ExtendedTransformation()
filter = SimpleFilter()
preprocessor.fit(X_train, y_train)
X_processed, y_processed = preprocessor.transform(X_train, y_train)
filter.fit(X_processed, y_processed)
X_filtered, y_filtered = filter.transform(X_processed, y_processed)

X shape:  (20974, 40)
bin_vars_columns shape:  (36,)
low_card_columns shape:  37
X shape:  (20974, 40)
X_low_card   shape:  (20974, 113)
X_high_card shape:  (20974, 50)
X_crossed_features shape:  (20974, 6670)
X_EXPANDED shape:  (20974, 6835)
(20974, 6835)
(20974, 4173)
(20974, 3193)
(20974, 1635)
(20974, 4173)
(20974, 3193)
(20974, 1635)


In [4]:
df_test = pd.read_csv("data/preprocessed/test_data.csv")
X_test, y_test = df_test.drop(columns=['Price']), df_test[['Price']]
X_test_proccesed, y_test_proccessed = preprocessor.transform(X_test, y_test)
X_test_filtered, y_test_filtered = filter.transform(X_test_proccesed, y_test_proccessed)

X shape:  (8989, 40)
X_low_card   shape:  (8989, 113)
X_high_card shape:  (8989, 50)
X_crossed_features shape:  (8989, 6670)
X_EXPANDED shape:  (8989, 6835)
(8989, 4173)
(8989, 3193)
(8989, 1635)


# 3. Optimización de hiperparámetros

Optmizamos con el quantile 0.5, Aunque posteriormente apliquemos los quantiles que consideremos para nuestro intervalo

In [8]:
# optimizamos para calcular el quantil medio con mejor precisión usando GradientBoostingRegressor.

import optuna
from sklearn.ensemble import GradientBoostingRegressor
import sklearn.model_selection
from sklearn.metrics import mean_pinball_loss, make_scorer

def objective(trial):
    x, y = X_filtered, y_filtered.flatten()

    n_estimators = trial.suggest_int("n_estimators", 10, 500, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 32, log=True)
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)

    estimator = GradientBoostingRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        min_samples_split=min_samples_split,
        loss="quantile",
        alpha=0.5,
        random_state=0
    )

    score = make_scorer(mean_pinball_loss, alpha=0.5)
    scoring = sklearn.model_selection.cross_val_score(estimator, x, y, n_jobs=-1, cv=3, scoring=score)

    return scoring.mean()

In [9]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)
print(study.best_trial)

[I 2025-05-19 19:28:10,391] A new study created in memory with name: no-name-409f6666-0cca-45b7-9930-16ae31b18423
[I 2025-05-19 19:35:03,498] Trial 0 finished with value: 0.24977541011775858 and parameters: {'n_estimators': 232, 'max_depth': 12, 'learning_rate': 0.04353592153806476, 'subsample': 0.7169899135520859, 'min_samples_split': 14}. Best is trial 0 with value: 0.24977541011775858.
[I 2025-05-19 19:38:26,032] Trial 1 finished with value: 0.25089210426108793 and parameters: {'n_estimators': 152, 'max_depth': 12, 'learning_rate': 0.06892265066925833, 'subsample': 0.544714178031843, 'min_samples_split': 19}. Best is trial 0 with value: 0.24977541011775858.
[I 2025-05-19 19:39:06,564] Trial 2 finished with value: 0.38443164329469276 and parameters: {'n_estimators': 43, 'max_depth': 6, 'learning_rate': 0.006950955561444311, 'subsample': 0.773194895439765, 'min_samples_split': 2}. Best is trial 0 with value: 0.24977541011775858.
[I 2025-05-19 19:40:17,126] Trial 3 finished with value:

FrozenTrial(number=0, state=1, values=[0.24977541011775858], datetime_start=datetime.datetime(2025, 5, 19, 19, 28, 10, 393298), datetime_complete=datetime.datetime(2025, 5, 19, 19, 35, 3, 498194), params={'n_estimators': 232, 'max_depth': 12, 'learning_rate': 0.04353592153806476, 'subsample': 0.7169899135520859, 'min_samples_split': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=True, low=10, step=1), 'max_depth': IntDistribution(high=32, log=True, low=3, step=1), 'learning_rate': FloatDistribution(high=0.1, log=True, low=0.001, step=None), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1)}, trial_id=0, value=None)


In [10]:
study.best_trial.params

{'n_estimators': 232,
 'max_depth': 12,
 'learning_rate': 0.04353592153806476,
 'subsample': 0.7169899135520859,
 'min_samples_split': 14}

In [11]:
estimator_params = study.best_trial.params
estimator_params['loss'] = "quantile"
estimator_params['alpha'] = 0.5
estimator_params

{'n_estimators': 232,
 'max_depth': 12,
 'learning_rate': 0.04353592153806476,
 'subsample': 0.7169899135520859,
 'min_samples_split': 14,
 'loss': 'quantile',
 'alpha': 0.5}

# 4. Configuración del estimador base

In [12]:
estimator = GradientBoostingRegressor(**estimator_params)

# 5. Configuración del modelo mappie basado en quantile regressor

In [13]:
alpha = 0.2 # 80% de confianza
quantile_params = {"method": "quantile", "cv": "split", "alpha": alpha}

In [15]:
mapie = MapieQuantileRegressor(estimator, **quantile_params)
mapie.fit(
            X_filtered, 
            y_filtered,
            calib_size=0.3,
            random_state=0
        )


c:\Users\HJ169WU\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\HJ169WU\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


MapieQuantileRegressor(alpha=0.2, cv='split',
                       estimator=GradientBoostingRegressor(alpha=0.5,
                                                           learning_rate=0.04353592153806476,
                                                           loss='quantile',
                                                           max_depth=12,
                                                           min_samples_split=14,
                                                           n_estimators=232,
                                                           subsample=0.7169899135520859))

In [ ]:
from mapie.regression import MapieQuantileRegressor
import pickle
import os

CHECKPOINTS_DIR = "checkpoints"

for alpha, name in zip([0.2, 0.1, 0.01], ["80", "90", "99"]):
    quantile_params = {"method": "quantile", "cv": "split", "alpha": alpha}
    mapie = MapieQuantileRegressor(estimator, **quantile_params)
    mapie.fit(
        X_filtered,
        y_filtered,
        calib_size=0.2,  # 20% para calibración
        random_state=0
    )
    with open(os.path.join(CHECKPOINTS_DIR, f"model_with_intervals_{name}.pkl"), "wb") as f:
        pickle.dump(mapie, f)

# 6. Predicción de los datos de test

In [11]:
y_pred, y_pis = mapie.predict(X_test_filtered)

INFO:root:The predictions are ill-sorted.
INFO:root:The predictions are ill-sorted.


In [12]:
y_pis[:,0]
preprocessor.inverse_transform(y_pis[:,0])

/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(


array([[2616183.78196681],
       [3427044.25554149],
       [2469686.86245631],
       ...,
       [2000000.        ],
       [9500000.        ],
       [2800175.99738977]])

In [13]:
# convert to original scale
y_mediam = preprocessor.inverse_transform(y_pred.reshape(-1,1))
y_low = preprocessor.inverse_transform(y_pis[:,0])
y_high = preprocessor.inverse_transform(y_pis[:,1])


/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(


# 7. Evaluación de cobertura y longitud media

In [15]:
# evaluamos su cobertura, para ver si realmente en el 80% de los casos el valor real está dentro del intervalo mostrado.
coverage = regression_coverage_score(y_test, y_low, y_high)
mean_width = regression_mean_width_score(y_low, y_high)

print(f"regresion coverage: {coverage}")
print(f"interval mean width: {mean_width}")

regresion coverage: 0.7970853265101792
interval mean width: 15865322.680892356


# 8. Guardar los modelos para su exportación a un entorno de serving/inferencia

In [16]:
import os
import pickle

CHECKPOINTS_DIR = "checkpoints"
# Save the objects
with open(os.path.join(CHECKPOINTS_DIR, "preprocessor.pkl"), "wb") as f:
    pickle.dump(preprocessor, f)

with open(os.path.join(CHECKPOINTS_DIR, "filter.pkl"), "wb") as f:
    pickle.dump(filter, f)

with open(os.path.join(CHECKPOINTS_DIR, "model_with_intervals.pkl"), "wb") as f:
    pickle.dump(mapie, f)

# 9. comprobar que se pueden recuperar los modelos y ejecutar correctamente

In [20]:
# Load the objects
with open(os.path.join(CHECKPOINTS_DIR, "preprocessor.pkl"), "rb") as f:
    my_preprocessor = pickle.load(f)

with open(os.path.join(CHECKPOINTS_DIR, "filter.pkl"), "rb") as f:
    my_filter = pickle.load(f)

with open(os.path.join(CHECKPOINTS_DIR, "model_with_intervals.pkl"), "rb") as f:
    model_w_intervals = pickle.load(f)

In [ ]:
# preprocesamos
X_processed, y_processed = my_preprocessor.transform(X_test, y_test)


X shape:  (8989, 40)
X_low_card   shape:  (8989, 113)
X_high_card shape:  (8989, 50)
X_crossed_features shape:  (8989, 6670)
X_EXPANDED shape:  (8989, 6835)


In [ ]:
# filtramos
X_filtered, y_filtered = my_filter.transform(X_processed, y_processed)

(8989, 4173)
(8989, 3193)
(8989, 1635)


In [ ]:
# predecimos
pred, intervals = model_w_intervals.predict(X_filtered)

INFO:root:The predictions are ill-sorted.
INFO:root:The predictions are ill-sorted.


In [ ]:
# transformamos a la escala adecuada.
y_mediam = my_preprocessor.inverse_transform(y_pred.reshape(-1,1))
y_low = my_preprocessor.inverse_transform(y_pis[:,0])
y_high = my_preprocessor.inverse_transform(y_pis[:,1])

/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(


In [27]:
# evaluamos su cobertura, para ver si realmente en el 80% de los casos el valor real está dentro del intervalo mostrado.
coverage = regression_coverage_score(y_test, y_low, y_high)
mean_width = regression_mean_width_score(y_low, y_high)

print(f"regresion coverage: {coverage}")
print(f"interval mean width: {mean_width}")

regresion coverage: 0.7970853265101792
interval mean width: 15865322.680892356
